# 04 - Feature Engineering

## Preparing Data for One-Hour-Ahead Extreme Demand Prediction

The previous analysis showed that extreme demand is related to recent demand behavior, short-term changes, operating time and equipment type.

In this notebook, I create features for predicting whether an asset will experience extreme demand one hour ahead.

All features are based only on information available at or before the prediction time.

In [2]:
import pandas as pd
import numpy as np

## 1. Load and Prepare the Dataset

I load the cleaned dataset, convert the timestamp to datetime and sort the observations by asset and time before creating time-dependent features.

In [3]:
df = pd.read_csv(
    "../data_processed/manufacturing_energy_clean.csv"
)

df["window_start_utc"] = pd.to_datetime(
    df["window_start_utc"]
)

df = df.sort_values(
    ["AssetId", "window_start_utc"]
).reset_index(drop=True)

print(df.shape)

(1016182, 16)


In [4]:
df[["AssetId", "asset_type", "window_start_utc", "Demand_kW"]].head()

,AssetId,asset_type,window_start_utc,Demand_kW
0,ahu_a,HVAC_AHU,2024-12-31 08:30:00,0.013069
1,ahu_a,HVAC_AHU,2024-12-31 08:45:00,0.014416
2,ahu_a,HVAC_AHU,2024-12-31 09:00:00,0.013955
3,ahu_a,HVAC_AHU,2024-12-31 09:15:00,0.013948
4,ahu_a,HVAC_AHU,2024-12-31 09:30:00,0.014930


## 2. Create the One-Hour-Ahead Prediction Target

The goal is to predict whether an asset will experience extreme demand exactly one hour in the future.

Because assets operate at different demand levels, extreme demand is defined separately for each asset using its 90th percentile demand threshold.

To avoid using future information, these thresholds are calculated from the training period only and then applied to the full dataset.

In [5]:
# Chronological split used in the ML pipeline
split_time = pd.Timestamp("2025-11-08 20:15:00")

# Calculate thresholds using training data only
threshold_data = df[
    df["window_start_utc"] < split_time
].copy()

thresholds = (
    threshold_data
    .groupby("AssetId")["Demand_kW"]
    .quantile(0.90)
)

# Apply the training thresholds to the full dataset
df["extreme_threshold"] = df["AssetId"].map(thresholds)

# Keep assets with a valid historical threshold
df = df[
    df["extreme_threshold"].notna()
].copy()

print("Assets included in modeling:", df["AssetId"].nunique())
print("Missing thresholds:", df["extreme_threshold"].isna().sum())

Assets included in modeling: 37
Missing thresholds: 0


In [6]:
df["is_extreme_now"] = (
    df["Demand_kW"] > df["extreme_threshold"]
).astype(int)

df["is_extreme_now"].value_counts(normalize=True) * 100

is_extreme_now
0    88.942258
1    11.057742
Name: proportion, dtype: float64

### Create the One-Hour-Ahead Target

With 15-minute observations, four rows ahead normally represents one hour. Because the dataset contains a small number of timestamp gaps, I also verify that the future observation is exactly one hour later.

In [7]:
df["future_time"] = (
    df.groupby("AssetId")["window_start_utc"]
    .shift(-4)
)

df["future_extreme"] = (
    df.groupby("AssetId")["is_extreme_now"]
    .shift(-4)
)

df["future_gap"] = (
    df["future_time"] - df["window_start_utc"]
)

df["future_gap"].value_counts().head(10)

future_gap
0 days 01:00:00    996357
0 days 01:15:00       780
0 days 01:30:00       237
0 days 01:45:00       161
0 days 07:30:00       154
0 days 09:45:00       144
0 days 02:00:00       125
0 days 02:15:00       113
0 days 02:30:00       106
0 days 03:15:00        87
Name: count, dtype: int64

### Keep Only Valid One-Hour-Ahead Observations

I keep only rows where the future observation occurs exactly one hour later. This ensures that the prediction target always represents the same one-hour horizon.

In [8]:
ml_df = df[
    df["future_gap"] == pd.Timedelta(hours=1)
].copy()

ml_df["extreme_1h_ahead"] = (
    ml_df["future_extreme"].astype(int)
)

print("Rows available for modeling:", len(ml_df))

ml_df["extreme_1h_ahead"].value_counts(normalize=True) * 100

Rows available for modeling: 996357


extreme_1h_ahead
0    88.918329
1    11.081671
Name: proportion, dtype: float64

## 3. Historical Demand Features

Recent demand is strongly related to current operating conditions. I create lag features representing demand 15, 30, 45 and 60 minutes before the prediction time.

In [9]:
df["demand_15m_ago"] = (
    df.groupby("AssetId")["Demand_kW"].shift(1)
)

df["demand_30m_ago"] = (
    df.groupby("AssetId")["Demand_kW"].shift(2)
)

df["demand_45m_ago"] = (
    df.groupby("AssetId")["Demand_kW"].shift(3)
)

df["demand_60m_ago"] = (
    df.groupby("AssetId")["Demand_kW"].shift(4)
)

In [10]:
lag_cols = [
    "demand_15m_ago",
    "demand_30m_ago",
    "demand_45m_ago",
    "demand_60m_ago"
]

ml_df[lag_cols] = df.loc[ml_df.index, lag_cols]

In [11]:
ml_df[
    ["AssetId", "window_start_utc", "Demand_kW"] + lag_cols
].head(10)

,AssetId,window_start_utc,Demand_kW,demand_15m_ago,demand_30m_ago,demand_45m_ago,demand_60m_ago
0,ahu_a,2024-12-31 08:30:00,0.013069,NaN,NaN,NaN,NaN
1,ahu_a,2024-12-31 08:45:00,0.014416,0.013069,NaN,NaN,NaN
2,ahu_a,2024-12-31 09:00:00,0.013955,0.014416,0.013069,NaN,NaN
3,ahu_a,2024-12-31 09:15:00,0.013948,0.013955,0.014416,0.013069,NaN
4,ahu_a,2024-12-31 09:30:00,0.014930,0.013948,0.013955,0.014416,0.013069
5,ahu_a,2024-12-31 09:45:00,0.015270,0.014930,0.013948,0.013955,0.014416
6,ahu_a,2024-12-31 10:00:00,0.015605,0.015270,0.014930,0.013948,0.013955
7,ahu_a,2024-12-31 10:15:00,0.015035,0.015605,0.015270,0.014930,0.013948
8,ahu_a,2024-12-31 10:30:00,0.014948,0.015035,0.015605,0.015270,0.014930
9,ahu_a,2024-12-31 10:45:00,0.014429,0.014948,0.015035,0.015605,0.015270


### Validate the Historical Time Window

Because the dataset contains a small number of timestamp gaps, four previous observations do not always represent exactly one hour. I therefore verify that the 60-minute lag comes from exactly one hour before the prediction time.

In [12]:
df["time_60m_ago"] = (
    df.groupby("AssetId")["window_start_utc"]
    .shift(4)
)

df["historical_gap"] = (
    df["window_start_utc"] - df["time_60m_ago"]
)

df["historical_gap"].value_counts().head(10)

historical_gap
0 days 01:00:00    996357
0 days 01:15:00       780
0 days 01:30:00       237
0 days 01:45:00       161
0 days 07:30:00       154
0 days 09:45:00       144
0 days 02:00:00       125
0 days 02:15:00       113
0 days 02:30:00       106
0 days 03:15:00        87
Name: count, dtype: int64

### Keep Valid Historical Windows

I keep only observations where the historical window covers exactly one hour so that the lag features represent consistent time intervals.

In [13]:
valid_history = (
    df["historical_gap"] == pd.Timedelta(hours=1)
)

ml_df = ml_df[
    valid_history.loc[ml_df.index]
].copy()

print("Rows after validating history:", len(ml_df))

Rows after validating history: 993411


## 4. Demand Change Features

Demand level alone does not show whether consumption is rising or falling. I therefore create features representing the change during the previous 15 minutes and the previous hour.

In [14]:
ml_df["change_15m"] = (
    ml_df["Demand_kW"] - ml_df["demand_15m_ago"]
)

ml_df["change_60m"] = (
    ml_df["Demand_kW"] - ml_df["demand_60m_ago"]
)

In [15]:
ml_df[
    [
        "Demand_kW",
        "demand_15m_ago",
        "demand_60m_ago",
        "change_15m",
        "change_60m"
    ]
].describe()

,Demand_kW,demand_15m_ago,demand_60m_ago,change_15m,change_60m
count,993411.000000,993411.000000,993411.000000,993411.000000,993411.000000
mean,11.847673,11.847928,11.847329,-0.000255,0.000344
std,38.965179,38.965832,38.968485,5.466717,8.613969
min,0.000000,0.000000,0.000000,-241.882785,-285.989675
25%,0.025188,0.025161,0.025044,-0.007382,-0.016048
50%,0.958108,0.958129,0.958123,0.000000,0.000000
75%,8.294905,8.296288,8.299368,0.005428,0.008764
max,591.314213,591.314213,591.314213,253.147544,323.638609


## 5. Rolling Demand Features

Lag features show demand at individual points in time. Rolling features summarize the recent operating state by measuring the average demand and variability during the previous hour.

In [16]:
rolling = (
    df.groupby("AssetId")["Demand_kW"]
    .rolling(window=4)
)

df["rolling_mean_1h"] = (
    rolling.mean()
    .reset_index(level=0, drop=True)
)

df["rolling_std_1h"] = (
    rolling.std()
    .reset_index(level=0, drop=True)
)

In [17]:
ml_df["rolling_mean_1h"] = df.loc[
    ml_df.index, "rolling_mean_1h"
]

ml_df["rolling_std_1h"] = df.loc[
    ml_df.index, "rolling_std_1h"
]

In [18]:
ml_df[
    ["rolling_mean_1h", "rolling_std_1h"]
].describe()

,rolling_mean_1h,rolling_std_1h
count,993411.000000,993411.000000
mean,11.848055,1.256651
std,38.767690,4.365250
min,0.000000,0.000000
25%,0.026994,0.000282
50%,1.044815,0.009834
75%,8.337900,0.574078
max,556.843239,155.727623


## 6. Relative Demand and Time Features

Because assets operate at different demand levels, I calculate current demand relative to each asset's extreme-demand threshold.

I also create time features to represent differences in operating patterns by hour and day of the week.

In [19]:
ml_df["demand_threshold_ratio"] = np.where(
    ml_df["extreme_threshold"] > 0,
    ml_df["Demand_kW"] / ml_df["extreme_threshold"],
    0
)

In [20]:
ml_df["DateLocal"] = pd.to_datetime(ml_df["DateLocal"])

ml_df["hour"] = ml_df["HourLocal"]
ml_df["day_of_week"] = ml_df["DateLocal"].dt.dayofweek
ml_df["is_weekend"] = (
    ml_df["day_of_week"] >= 5
).astype(int)

In [21]:
ml_df[
    [
        "demand_threshold_ratio",
        "hour",
        "day_of_week",
        "is_weekend"
    ]
].describe()

print(
    "Infinite threshold ratios:",
    np.isinf(ml_df["demand_threshold_ratio"]).sum()
)

Infinite threshold ratios: 0


## 7. Final Feature Set and Leakage Check

I define the final predictor variables and verify that variables used to construct the future target are not included as model features.

This ensures that the model only receives information available at prediction time.

In [22]:
features = [
    "Demand_kW",
    "demand_15m_ago",
    "demand_30m_ago",
    "demand_45m_ago",
    "demand_60m_ago",
    "change_15m",
    "change_60m",
    "rolling_mean_1h",
    "rolling_std_1h",
    "demand_threshold_ratio",
    "hour",
    "day_of_week",
    "is_weekend",
    "AssetId",
    "asset_type"
]

target = "extreme_1h_ahead"

print("Number of features:", len(features))

Number of features: 15


In [23]:
leakage_columns = [
    "future_time",
    "future_extreme",
    "future_gap"
]

set(features).intersection(leakage_columns)

set()

## 8. Save the ML-Ready Dataset

The final dataset contains the engineered features, one-hour-ahead target, timestamp and asset information needed for time-based model development.

In [24]:
final_columns = (
    ["window_start_utc"] +
    features +
    [target]
)

model_data = ml_df[final_columns].copy()

print("Final modeling shape:", model_data.shape)
model_data.head()

Final modeling shape: (993411, 17)


,window_start_utc,Demand_kW,demand_15m_ago,demand_30m_ago,demand_45m_ago,demand_60m_ago,change_15m,change_60m,rolling_mean_1h,rolling_std_1h,demand_threshold_ratio,hour,day_of_week,is_weekend,AssetId,asset_type,extreme_1h_ahead
4,2024-12-31 09:30:00,0.014930,0.013948,0.013955,0.014416,0.013069,0.000983,0.001861,0.014312,0.000467,0.001503,9,1,0,ahu_a,HVAC_AHU,0
5,2024-12-31 09:45:00,0.015270,0.014930,0.013948,0.013955,0.014416,0.000339,0.000854,0.014526,0.000677,0.001537,9,1,0,ahu_a,HVAC_AHU,0
6,2024-12-31 10:00:00,0.015605,0.015270,0.014930,0.013948,0.013955,0.000336,0.001650,0.014938,0.000716,0.001571,10,1,0,ahu_a,HVAC_AHU,0
7,2024-12-31 10:15:00,0.015035,0.015605,0.015270,0.014930,0.013948,-0.000571,0.001087,0.015210,0.000299,0.001514,10,1,0,ahu_a,HVAC_AHU,0
8,2024-12-31 10:30:00,0.014948,0.015035,0.015605,0.015270,0.014930,-0.000087,0.000017,0.015214,0.000294,0.001505,10,1,0,ahu_a,HVAC_AHU,0


In [25]:
model_data.isna().sum().sort_values(ascending=False)

window_start_utc          0
Demand_kW                 0
demand_15m_ago            0
demand_30m_ago            0
demand_45m_ago            0
demand_60m_ago            0
change_15m                0
change_60m                0
rolling_mean_1h           0
rolling_std_1h            0
demand_threshold_ratio    0
hour                      0
day_of_week               0
is_weekend                0
AssetId                   0
asset_type                0
extreme_1h_ahead          0
dtype: int64

In [26]:
model_data.to_csv(
    "../data_processed/ml_ready_energy_data.csv",
    index=False
)

## 9. Summary

This notebook prepared the dataset for one-hour-ahead extreme-demand prediction.

- Asset-specific extreme-demand thresholds were calculated using the training period only.
- Only observations with valid one-hour historical and future windows were retained.
- Historical demand features represent demand during the previous hour.
- Change and rolling features describe recent demand behavior.
- Relative-demand features account for differences between assets.
- Time and equipment information represent different operating conditions.
- Future target-construction variables were excluded from the model features.

The final dataset is now ready for time-based machine learning.